In [147]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    display(f"Loaded config from: {config_path}")
    display(f"Base URL: {api_base_url}")
    display(f"Access ID: {api_access_id}")
    display(f"Default Org: {api_default_org}")
except Exception as e:
    display(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    display("ThreatConnect initialized.")
except Exception as e:
    display(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    display("RequestObject successfully created.")
except Exception as e:
    display(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)




'Loaded config from: C:\\Users\\jaskew\\Documents\\project_repository\\scripts\\Data Movement\\ThrearConnect-api-pull\\utils\\config.json'

'Base URL: https://hvs.threatconnect.com/api'

'Access ID: 09783848890162390382'

'Default Org: HTOC Org'

'ThreatConnect initialized.'

'RequestObject successfully created.'

In [148]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=3)).date()
start = f"{start_date}T00:00:00Z"
type_names = ["Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
              "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])
list_of_owners = ['HTOC Org']
final_results = []

# Query indicators
for owner in list_of_owners:
    display(f"Querying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition}) AND '
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators?tql={tql_encoded}&fields=tags,observations,associatedGroups,falsePositives,&resultStart=0&resultLimit=10000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
    except Exception as e:
        display(f"Failed to query indicators for {owner}: {e}")

# Normalize results
normalized_data = []
for result in final_results:
    data_items = result.get('data', [])
    if not data_items:
        display("No data returned in API response:", result)
    for item in data_items:
        if isinstance(item, dict) and 'summary' in item:
            normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    observed_src['indicator'] = observed_src['summary'].str.split().str[0].str.strip()
    observed_src.drop_duplicates(subset='indicator', inplace=True)
    observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True)
    observed_src = observed_src[observed_src['lastObserved'] >= pd.to_datetime(start)]
else:
    display("No valid indicator data found.")
    observed_src = pd.DataFrame()

display(observed_src)


'Querying owner: HTOC Org'

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,associatedGroups.data,hostName,dnsActive,whoisActive,source,lastFalsePositive,md5,url,address,indicator
0,6755399467420975,2025-08-13T15:08:24Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-10T11:57:55Z,3.0,76,RITM0606742,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45.141.56.48
1,6755399459033730,2025-06-16T17:42:19Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-10T11:57:01Z,3.0,68,RITM0589984,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,109.70.100.1
2,6755399459033718,2025-06-16T17:42:07Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-10T11:57:01Z,3.0,68,RITM0589984,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,109.70.100.68
3,6755399447111249,2025-05-14T17:44:50Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-10T11:56:01Z,3.0,64,FBI Email Alert May 14 2025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,87.236.176.223
4,6755399463130395,2025-07-12T16:00:58Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-10T11:55:59Z,3.0,72,INC9143670,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,125.124.50.87
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1235,4476331,2023-11-16T13:43:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-17T23:24:44Z,4.0,85,NaN,...,"[{'id': 291847, 'dateAdded': '2023-12-15T13:16...",NaN,NaN,NaN,http://selligenttier.naylorcampaigns.com,NaN,NaN,selligenttier.naylorcampaigns.com,NaN,selligenttier.naylorcampaigns.com
1236,4324196,2023-04-21T14:22:00Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-16T23:23:57Z,3.0,84,NaN,...,"[{'id': 149988, 'dateAdded': '2023-04-21T14:08...",NaN,NaN,NaN,NaN,NaN,NaN,geo.netsupportsoftware.com/location/loca.asp,NaN,geo.netsupportsoftware.com/location/loca.asp
1237,4442727,2023-10-06T23:26:48Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-03-28T23:23:43Z,3.0,83,NaN,...,NaN,NaN,NaN,NaN,https://google,NaN,NaN,google,NaN,google
1238,4517160,2024-02-05T17:10:45Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-03-05T23:23:34Z,3.0,67,NaN,...,"[{'id': 313616, 'dateAdded': '2024-02-05T16:18...",NaN,NaN,NaN,https://realinvestmentadvice.com/,NaN,NaN,realinvestmentadvice.com/,NaN,realinvestmentadvice.com/


In [155]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=3)).date()
start = f"{start_date}T00:00:00Z"

# Fields to return from GET /v3/indicators/{indicatorId or indicator}
detail_fields = [
    "ownerName", "type", "summary", "dateAdded", "lastModified", "active",
    "confidence", "rating", "threatAssessScore", "observations",
    "tags", "attributes", "securityLabels", "associatedGroups"
]

list_of_indicators = ['207.167.64.24'] 

# Build the keys to fetch
if "observed_src" in globals() and isinstance(observed_src, pd.DataFrame) and not observed_src.empty:
    if "id" in observed_src.columns:
        indicator_keys = (
            observed_src["id"].dropna().astype(str).str.strip().tolist()
        )
    elif "summary" in observed_src.columns:
        indicator_keys = (
            observed_src["summary"].dropna().astype(str).str.strip().tolist()
        )
    else:
        indicator_keys = list_of_indicators
else:
    indicator_keys = list_of_indicators

# De-duplicate while preserving order
seen = set()
ordered_keys = []
for k in indicator_keys:
    if k and k not in seen:
        seen.add(k)
        ordered_keys.append(k)

final_results = []

# Query indicators (single-resource endpoint only)
for key in ordered_keys:
    display(f"Querying indicator: {key}")
    try:
        # URL-encode the path segment (works for both numeric IDs and summary strings)
        path_key = urllib.parse.quote(str(key), safe="")
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators/{path_key}?fields=tags,observations,associatedGroups&resultStart=0&resultLimit=10000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            result = response.json() or {}
            final_results.append(result)
    except Exception as e:
        pass

# Normalize results
normalized_data = []
for result in final_results:
    data_item = result.get('data', {})
    # v3 single GET usually returns a dict; occasionally data['indicator'] may hold it
    if isinstance(data_item, dict):
        record = data_item.get('indicator', data_item)
        if isinstance(record, dict) and 'summary' in record:
            normalized_data.append(record)
    elif isinstance(data_item, list):
        # Very rare shape; keep parity with earlier normalizer
        for item in data_item:
            if isinstance(item, dict) and 'summary' in item:
                normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    # Derive a simple 'indicator' token from summary (e.g., first token)
    if 'summary' in observed_src.columns:
        observed_src['indicator'] = observed_src['summary'].astype(str).str.split().str[0].str.strip()

    # Drop dupes on 'indicator' if it exists
    if 'indicator' in observed_src.columns:
        observed_src.drop_duplicates(subset='indicator', inplace=True)

    # Filter to lastObserved >= start, if lastObserved exists
    if 'lastObserved' in observed_src.columns:
        observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True, errors='coerce')
        #observed_src = observed_src[observed_src['lastObserved'] >= pd.to_datetime(start)]
else:
    display("No valid indicator data found.")
    observed_src = pd.DataFrame()

display(observed_src)


'Querying indicator: 5629499536037723'

'Querying indicator: 6755399443027790'

'Querying indicator: 5629499564010922'

'Querying indicator: 5269147'

'Querying indicator: 5629499542020635'

...

'Querying indicator: 6755399448037607'

'Querying indicator: 4193568'

'Querying indicator: 6755399460008411'

'Querying indicator: 6755399460007994'

KeyboardInterrupt: 

In [132]:
# Unnest tags.data in observed_src to get a list of each tag per indicator

# Explode tags.data to one row per tag
tags_exploded = (
    observed_src[['indicator', 'tags.data']]
      .explode('tags.data')
      .dropna(subset=['tags.data'])
)

# Extract tag name from each tag object
tags_exploded['tag_name'] = tags_exploded['tags.data'].apply(lambda x: x.get('name') if isinstance(x, dict) else None)

# Aggregate all tag names into a list per indicator
tags_per_indicator = (
    tags_exploded.groupby('indicator')['tag_name']
    .apply(lambda x: [t for t in x if t])
    .reset_index()
    .rename(columns={'tag_name': 'tag_list'})
)

# Merge back to observed_src if desired
observed_src_with_tags = observed_src.merge(tags_per_indicator, on='indicator', how='left')

display(observed_src_with_tags[['indicator', 'tag_list']])

,indicator,tag_list
0,141.98.10.225,"[OS Splunk API, VA CSOC CTS Splunk, Observed, ..."
1,123.150.138.197,"[OS Splunk API, FDA Splunk API, suspicious, CM..."
2,122.231.191.3,"[DHA Splunk API, OS Splunk API, FDA Splunk API..."
3,23.120.100.230,"[FortiGate, DHA Splunk API, Brute Force, forti..."
4,104.21.43.80,"[Play, Play ransomware group, VA CSOC CTS Splu..."
...,...,...
1253,click.email.active.com,"[OS Splunk API, htoc_wl, FDA Splunk API, CMS S..."
1254,careersinpsychology.org,"[Malware Family: GOOTLOADER, Gootloader]"
1255,aka.ms/o0ukef,"[FDA Splunk API, Observed, Phishing Email, inv..."
1256,selligenttier.naylorcampaigns.com,"[OS Splunk API, VA OIS CSOC CTS Splunk, FDA Sp..."


In [139]:
# List of tags to count (case-insensitive, strip whitespace)
tags_of_interest = [
    "Scanning", "DDoS", "Spam", "Phishing", "Cryptojacking",
    "Credential Stuffing", "Ransomware", "Data Theft",
    "Cross Site Scripting Attacks", "SQL Injections"
]

# Normalize tag_list to lowercase for matching
def tag_counter(tag_list, tag):
    if not isinstance(tag_list, list):
        return 0
    return sum(1 for t in tag_list if isinstance(t, str) and t.strip().lower() == tag.lower())

tag_counts = {}
for tag in tags_of_interest:
    tag_counts[tag] = observed_src_with_tags['tag_list'].apply(lambda lst: tag_counter(lst, tag)).sum()
    # Add a column to observed_src with the list of matching "Botnet" tags per indicator
    botnet_tags = set(tags_of_interest)
    def extract_botnet_tags(tag_list):
        if not isinstance(tag_list, list):
            return []
        return [t for t in tag_list if isinstance(t, str) and t.strip().lower() in {b.lower() for b in botnet_tags}]

    # Map indicator to its tag_list for efficient lookup
    indicator_to_tags = dict(zip(observed_src_with_tags['indicator'], observed_src_with_tags['tag_list']))

    observed_src['Botnet'] = observed_src['indicator'].map(lambda ind: extract_botnet_tags(indicator_to_tags.get(ind, [])))
# Display as a DataFrame for readability
pd.DataFrame(list(tag_counts.items()), columns=['Tag', 'Count'])

,Tag,Count
0,Scanning,0
1,DDoS,137
2,Spam,0
3,Phishing,15
4,Cryptojacking,0
5,Credential Stuffing,0
6,Ransomware,10
7,Data Theft,12
8,Cross Site Scripting Attacks,0
9,SQL Injections,0


In [46]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        display("No files found for the specified date range.")
    else:
        display(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            display(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        display(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=1)

# Load observed data
observed_data_df = load_observed_data(file_paths)



C:\Users\jaskew\AppData\Local\Temp\ipykernel_24276\1319479530.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


"Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250827.csv']"

'Loaded data from 1 files.'

In [134]:
import pandas as pd

df = observed_src.copy()

#df = df[df['indicator'].isin(observed_data_df['indicator'])]
# --- FULL TAGS UNNEST + PARTNERS ---

# explode tags.data
tags_exploded = (
    df[['indicator', 'tags.data']]
      .explode('tags.data')
      .dropna(subset=['tags.data'])
)

# normalize all fields in tags.data into flat columns
tags_norm = pd.json_normalize(tags_exploded['tags.data'])
tags_norm.columns = [f"tag_{c}" for c in tags_norm.columns]

# re‑attach indicator
tags_expanded = tags_exploded.reset_index(drop=True).join(tags_norm)

# extract partners
tags_expanded['partner'] = tags_expanded['tag_name'].map(
    lambda n: n[:-len(' Splunk API')] if isinstance(n, str) and n.endswith(' Splunk API') else None
)

# aggregate each tag_* field into a list per indicator
tag_fields = list(tags_norm.columns)
tag_agg = (
    tags_expanded
      .groupby('indicator', as_index=False)
      .agg({
          **{f: list for f in tag_fields},
          'partner': lambda x: [p for p in dict.fromkeys(x) if p]
      })
      .rename(columns={'partner':'partners'})
)

# --- GROUPS VIA EXPLODE + GROUPBY ---

#groups_exploded = (
#    df[['indicator', 'associatedGroups.data']]
#      .explode('associatedGroups.data')
#      .dropna(subset=['associatedGroups.data'])
#)

#group_norm = pd.json_normalize(
#    groups_exploded['associatedGroups.data']
#).rename(columns={'id':'group_id','name':'group_name'})

#groups_exploded = groups_exploded.reset_index(drop=True).join(group_norm)

#group_agg = (
#    groups_exploded
#      .drop_duplicates(subset=['indicator','group_id','group_name'])
#      .groupby('indicator', as_index=False)
#      .agg({
#          'group_id':   lambda ids: list(ids),
#          'group_name': lambda names: list(names)
#      })
#      .rename(columns={'group_id':'group_ids','group_name':'group_names'})
#)

# --- CORE AGGREGATION OF OTHER COLUMNS ---

first_cols = [
    'id','dateAdded','ownerId','ownerName','webLink','type','lastModified', 'falsePositives',
    'rating','confidence','description','summary','observations',
    'lastObserved','privateFlag','active','activeLocked','ip',
    'legacyLink','source','address','url'
]
# Remove 'hostName', 'dnsActive', 'whoisActive' from first_cols to avoid KeyError

# Add 'Botnet' column from observed_src if it exists
if 'Botnet' in observed_src.columns:
    df['Botnet'] = observed_src['Botnet']
    first_cols.append('Botnet')

base_agg = (
    df
      .drop(columns=[
          'createdBy.id','createdBy.userName','createdBy.firstName','createdBy.lastName',
          'createdBy.pseudonym','createdBy.owner','xid','eventType','documentDateAdded',
          'documentType','fileSize','fileName','downVoteCount','upVoteCount','type_group',
          'webLink_group','ownerName_group','ownerId_group','dateAdded_group','id_group',
          'platforms.count','tactics.count',
      ], errors='ignore')
      .groupby('indicator', as_index=False)[
          [c for c in first_cols if c in df.columns]
      ]
      .first()
)

# --- MERGE EVERYTHING BACK ---

agg_df = (
    base_agg
      .merge(tag_agg,   on='indicator', how='left')
#      .merge(group_agg, on='indicator', how='left')
)

def clean_list(lst):
    if not isinstance(lst, list):
        return []
    cleaned = []
    for v in lst:
        # drop any null‑like values
        try:
            if pd.isna(v):
                continue
        except Exception:
            pass
        # drop empty strings
        if isinstance(v, str) and v == "":
            continue
        cleaned.append(v)
    return cleaned

def list_to_csv(lst):
    """
    Takes a cleaned list and returns:
      - a comma-separated string of its items, OR
      - an empty string if there are none.
    """
    if not lst:
        return ""
    return ", ".join(str(v) for v in lst)

# apply to all your formerly‑list columns
#for col in ['partners','group_ids','group_names'] + tag_fields:
#    agg_df[col] = agg_df[col].apply(clean_list).apply(list_to_csv)

# Only apply to columns that exist in agg_df to avoid KeyError
for col in ['partners', 'group_ids', 'group_names'] + tag_fields:
    if col in agg_df.columns:
        agg_df[col] = agg_df[col].apply(clean_list).apply(list_to_csv)
    
display(agg_df)



,indicator,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,falsePositives,rating,...,tag_id,tag_name,tag_description,tag_lastUsed,tag_techniqueId,tag_tactics.data,tag_tactics.count,tag_platforms.data,tag_platforms.count,partners
0,1.192.18.4,6755399447111259,2025-05-14T17:47:35Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-01T08:25:44Z,0,3.0,...,"35760, 35057, 32811, 30479, 23576","OS Splunk API, FDA Splunk API, suspicious, CMS...",Observations reported by the HHS Ofice of the ...,"2025-09-04T13:20:32Z, 2025-09-04T04:58:05Z, 20...",,,,,,"OS, FDA, CMS"
1,101.89.148.7,5629499563086669,2025-08-09T18:13:37Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-04T03:35:29Z,0,3.0,...,"471298, 35760, 35057, 32811, 30479, 23667, 236...","DHA Splunk API, OS Splunk API, FDA Splunk API,...",Observations reported by the HHS Ofice of the ...,"2025-09-04T11:13:42Z, 2025-09-04T13:20:32Z, 20...",,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS"
2,101.89.174.236,5629499542017370,2025-05-21T18:39:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-04T09:22:52Z,0,3.0,...,"471298, 35760, 35057, 30479, 23667, 23630, 235...","DHA Splunk API, OS Splunk API, FDA Splunk API,...",Observations reported by the HHS Ofice of the ...,"2025-09-04T11:13:42Z, 2025-09-04T13:20:32Z, 20...",,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS, HHS"
3,102.0.5.152,6755399460008259,2025-07-02T12:05:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-30T09:02:28Z,0,4.0,...,"1469320, 1455870, 505200, 471298, 23769, 23576...","INDICATOR NOTICE 25178.1, Mr Hamza Group, T-Su...",,"2025-07-18T13:37:52Z, 2025-08-26T18:22:50Z, 20...",,,,,,DHA
4,102.209.18.96,6755399460007695,2025-07-02T12:05:31Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-03T10:41:41Z,0,4.0,...,"1469320, 1455870, 505200, 35057, 23769, 23576,...","INDICATOR NOTICE 25178.1, Mr Hamza Group, T-Su...",,"2025-07-18T13:37:52Z, 2025-08-26T18:22:50Z, 20...",,,,,,FDA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1253,www.chemdiv.com,5629499559316606,2025-07-14T15:36:16Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-09-03T07:02:04Z,0,5.0,...,"471298, 464075, 37189, 37188, 23769, 23630, 23...","DHA Splunk API, Secret Blizzard, UNC638, Venom...",,"2025-09-04T11:13:42Z, 2025-07-14T00:00:00Z, 20...",,,,,,"DHA, NIH"
1254,www.deepseek.com,5271608,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-09-02T11:46:02Z,0,3.0,...,"471298, 461545, 35760, 35752, 35057, 30770, 30...","DHA Splunk API, DeepSeek, OS Splunk API, VA OI...",Observations reported by the HHS Ofice of the ...,"2025-09-04T11:13:42Z, 2025-01-31T14:04:11Z, 20...",,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS"
1255,www.deepseek.com.cdn.cloudflare.net,5271612,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-09-03T11:56:56Z,0,3.0,...,"471298, 461545, 35752, 30479, 23769, 23630, 23...","DHA Splunk API, DeepSeek, VA OIS CSOC CTS Splu...",,"2025-09-04T11:13:42Z, 2025-01-31T14:04:11Z, 20...",,,,,,"DHA, CMS, NIH, IHS"
1256,www.sthda.com,4565837,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-09-04T06:50:06Z,0,4.0,...,"471298, 35760, 35752, 35057, 30770, 30479, 237...","DHA Splunk API, OS Splunk API, VA OIS CSOC CTS...",Observations reported by the HHS Ofice of the ...,"2025-09-04T11:13:42Z, 2025-09-04T13:20:32Z, 20...",,,,,,"DHA, OS, FDA, CMS, HRSA, NIH, IHS"


In [135]:
# ──────────────────────────────────────────────────────────────────────────────
# Enrich only final filtered indicators (Shodan & VirusTotalV3) and flatten
# ──────────────────────────────────────────────────────────────────────────────
import urllib.parse
import pandas as pd

COL_PATH = "data.enrichment.data"   # adjust if API changes

indicator_values = (
    agg_df["summary"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

display(f"Enriching {len(indicator_values)} indicators with Shodan & VirusTotalV3...")

enriched_results = []
failed = []

for value in indicator_values:
    try:
        encoded_value = urllib.parse.quote(value, safe="")
        q = urllib.parse.urlencode({"type": ["Shodan", "VirusTotalV3"]}, doseq=True)
        enrich_url = f"/v3/indicators/{encoded_value}/enrich?{q}"

        # Build a fresh RequestObject each loop (adjust to your SDK)
        ro = RequestObject()
        ro.set_http_method("POST")
        ro.set_request_uri(enrich_url)
        ro.set_body({})

        resp = tc.api_request(ro)
        resp.raise_for_status()

        data = resp.json()
        data["summary"] = value
        enriched_results.append(data)

    except Exception as e:
        failed.append({"summary": value, "error": str(e)})

# If nothing enriched, just carry on
if not enriched_results:
    display("No enrichment data retrieved.")
    recent_tags = agg_df.copy()

else:
    # One row per summary from enrichment payload
    df_enriched = (
        pd.json_normalize(enriched_results)
          .drop_duplicates(subset="summary", keep="last")
    )

    # Merge enrichment block into base
    recent_tags = agg_df.merge(df_enriched, on="summary", how="left")

    # ── Flatten enrichment payload without creating duplicate base rows ───────
    if COL_PATH in recent_tags.columns:
        exploded = (
            recent_tags[["summary", COL_PATH]]
            .explode(COL_PATH)
            .dropna(subset=[COL_PATH])
        )

        enrich_flat = pd.json_normalize(exploded[COL_PATH]).add_prefix("enrich_")
        enrich_flat["summary"] = exploded["summary"].values

        # ---- Aggregation helpers ---------------------------------------------
        def _flatten_lists(values):
            """Flatten one level of lists in a sequence, keep dicts intact."""
            out = []
            for v in values:
                if isinstance(v, list):
                    out.extend(v)
                else:
                    out.append(v)
            return out

        def _agg_obj(series: pd.Series):
            vals = [v for v in series.dropna()]
            if not vals:
                return None
            flat = _flatten_lists(vals)
            # if everything is hashable & not dict/list, dedupe
            if all(not isinstance(v, (list, dict)) for v in flat):
                return list(pd.unique(flat))
            return flat  # keep as-is when dicts/lists present

        obj_cols = enrich_flat.select_dtypes("object").columns.difference(["summary"])
        num_cols = enrich_flat.columns.difference(obj_cols.union({"summary"}))

        agg_dict = {c: _agg_obj for c in obj_cols}
        # choose your numeric aggregation; 'max' or 'first'
        agg_dict.update({c: "max" for c in num_cols})

        enrich_wide = (
            enrich_flat
              .groupby("summary", as_index=False)
              .agg(agg_dict)
        )

        # Remove raw list col and merge flattened cols back
        recent_tags = (
            recent_tags.drop(columns=[COL_PATH], errors="ignore")
                       .drop_duplicates(subset="summary")
                       .merge(enrich_wide, on="summary", how="left")
        )

        display(f"Successfully enriched and merged {len(df_enriched)} indicators.")
    else:
        display("Enrichment column not found; nothing to flatten.")

# Optional: report/log failures
if failed:
    display(f"{len(failed)} indicators failed to enrich.")
    # Example: pd.DataFrame(failed).to_json("enrich_failures.json", orient="records")


'Enriching 1258 indicators with Shodan & VirusTotalV3...'

Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host 3-get.njalla.fo cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"Request could not be completed due to a NullPointerException","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"Request could not be completed due to a NullPointerException","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL accentcare.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL aka.ms/o0ukef cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host api.deepseek.com cannot be 

'Successfully enriched and merged 1180 indicators.'

'78 indicators failed to enrich.'

In [140]:
recent_tags.drop(columns=['tag_id', 'tag_lastUsed', 'tag_lastModified', 'tag_ownerId', 
                          'tag_ownerName', 'tag_dateAdded', 'tag_description','tag_tactics.count', 
                          'tag_platform.data', 'tag_platform.count', 'data.id', 'data.dateAdded', 'data.ownerId',
                          'data.webLink', 'data.ownerName', 'data.lastModified', 'data.summary', 'data.ip',
                          'data.legacyLink','data.source', 'enrich_cloudProvider', 'enrich_cloudRegion', 'enrich_type',
                          'id'], inplace=True, errors='ignore')

recent_tags

,indicator,dateAdded,ownerId,ownerName,webLink,type,lastModified,falsePositives,rating,confidence,...,enrich_city,enrich_country,enrich_domains,enrich_hostNames,enrich_isp,enrich_openPorts,enrich_org,enrich_tags,enrich_vulnerabilities,enrich_vtMaliciousCount
0,1.192.18.4,2025-05-14T17:47:35Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-01T08:25:44Z,0,3.0,65,...,None,None,None,None,None,None,None,None,None,4.0
1,101.89.148.7,2025-08-09T18:13:37Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-04T03:35:29Z,0,3.0,77,...,[Shanghai],[China],None,None,[China Telecom (Group)],"[{'transport': 'tcp', 'port': 9999, 'data': 'H...",[CHINANET SHANGHAI PROVINCE NETWORK],None,None,10.0
2,101.89.174.236,2025-05-21T18:39:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-04T09:22:52Z,0,3.0,66,...,[Shanghai],[China],None,None,[China Telecom (Group)],"[{'transport': 'tcp', 'port': 22, 'data': 'SSH...",[CHINANET SHANGHAI PROVINCE NETWORK],[database],None,6.0
3,102.0.5.152,2025-07-02T12:05:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-30T09:02:28Z,0,4.0,65,...,[Nairobi],[Kenya],[airtelkenya.com],[152-5-0-102.r.airtelkenya.com],[Airtel Networks Kenya Limited],"[{'transport': 'udp', 'port': 161, 'product': ...",[Allocated to Airtel Kenya Mobile Broadband an...,[vpn],None,1.0
4,102.209.18.96,2025-07-02T12:05:31Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-03T10:41:41Z,0,4.0,65,...,[Nairobi],[Kenya],[veenet.africa],[static-97.veenet.africa],[VENNET SOLUTIONS LIMITED],"[{'transport': 'tcp', 'port': 80, 'data': 'HTT...",[VENNET SOLUTIONS LIMITED],[vpn],None,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1253,www.chemdiv.com,2025-07-14T15:36:16Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-09-03T07:02:04Z,0,5.0,93,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1254,www.deepseek.com,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-09-02T11:46:02Z,0,3.0,79,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1255,www.deepseek.com.cdn.cloudflare.net,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-09-03T11:56:56Z,0,3.0,77,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1256,www.sthda.com,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-09-04T06:50:06Z,0,4.0,45,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [141]:
from pymongo import MongoClient

# Connect to MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["Threat_Assess_Controlled_Scoring_Data"]
collection = db["controlled_enriched_observation_Data"]

# Convert DataFrame to dictionary records
records = recent_tags.to_dict(orient="records")

# Insert records into MongoDB
result = collection.insert_many(records)
print(f"Inserted {len(result.inserted_ids)} documents into enriched_observation_Data.")

client.close()


Inserted 1258 documents into enriched_observation_Data.


In [1]:
from pymongo import MongoClient
import pandas as pd

# Connect to MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["Threat_Assess_Controlled_Scoring_Data"]
collection = db["controlled_enriched_observation_Data"]

# Pull data back from MongoDB
docs = list(collection.find())
recent_tags = pd.DataFrame(docs)
print(f"Pulled {len(recent_tags)} documents from enriched_observation_Data.")


Pulled 3945 documents from enriched_observation_Data.


In [2]:
#recent_tags = recent_tags.head(50)
recent_tags.loc[recent_tags['enrich_vtMaliciousCount'].isnull(), 'enrich_vtMaliciousCount'] = 0
display(recent_tags)

,_id,indicator,dateAdded,ownerId,ownerName,webLink,type,lastModified,falsePositives,rating,...,enrich_city,enrich_country,enrich_domains,enrich_hostNames,enrich_isp,enrich_openPorts,enrich_org,enrich_tags,enrich_vulnerabilities,enrich_vtMaliciousCount
0,68af160dc60b14cf6da41a47,1.192.18.4,2025-05-14T17:47:35Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-25T07:41:56Z,0,3.0,...,None,None,None,None,None,None,None,None,None,4.0
1,68af160dc60b14cf6da41a48,100.27.42.247,2025-08-11T15:16:09Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-25T07:42:04Z,0,3.0,...,None,None,None,None,None,None,None,None,None,0.0
2,68af160dc60b14cf6da41a49,101.89.148.7,2025-08-09T18:13:37Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-25T02:03:09Z,0,3.0,...,[Shanghai],[China],None,None,[China Telecom (Group)],"[{'transport': 'tcp', 'port': 9999, 'data': 'H...",[CHINANET SHANGHAI PROVINCE NETWORK],None,None,11.0
3,68af160dc60b14cf6da41a4a,101.89.174.236,2025-05-21T18:39:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-26T08:07:02Z,0,3.0,...,[Shanghai],[China],None,None,[China Telecom (Group)],"[{'transport': 'tcp', 'port': 22, 'data': 'SSH...",[CHINANET SHANGHAI PROVINCE NETWORK],None,None,8.0
4,68af160dc60b14cf6da41a4b,102.0.5.152,2025-07-02T12:05:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-08-25T13:37:36Z,0,4.0,...,[Nairobi],[Kenya],[airtelkenya.com],[152-5-0-102.r.airtelkenya.com],[Airtel Networks Kenya Limited],"[{'transport': 'udp', 'port': 161, 'product': ...",[Allocated to Airtel Kenya Mobile Broadband an...,[vpn],None,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3940,68b9cf59c60b14cf6da429b2,www.chemdiv.com,2025-07-14T15:36:16Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-09-03T07:02:04Z,0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3941,68b9cf59c60b14cf6da429b3,www.deepseek.com,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-09-02T11:46:02Z,0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3942,68b9cf59c60b14cf6da429b4,www.deepseek.com.cdn.cloudflare.net,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-09-03T11:56:56Z,0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3943,68b9cf59c60b14cf6da429b5,www.sthda.com,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-09-04T06:50:06Z,0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [3]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=365)

# Load observed data
observed_data_df = load_observed_data(file_paths)



C:\Users\jaskew\AppData\Local\Temp\ipykernel_22972\1921769075.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250924.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250923.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250922.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250921.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250920.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250919.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250918.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250917.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250916.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250915.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250914.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250913.csv', 'Z:/HTOC/Data_Analytics/Data/Op

In [4]:
# Aggregate observed_data_df by 'indicator', counting unique obs_date occurrences
agg_by_indicator = (
    observed_data_df
    .groupby('indicator', as_index=False)['obs_date']
    .nunique()
    .rename(columns={'obs_date': 'obs_days_count'})
)

agg_by_indicator = agg_by_indicator[agg_by_indicator['indicator'].isin(recent_tags['indicator'])]
# Map obs_days_count from agg_by_indicator to recent_tags by indicator, rename to obs_count
recent_tags = recent_tags.merge(
    agg_by_indicator.rename(columns={'obs_days_count': 'obs_count'}),
    on='indicator',
    how='left'
)
display(agg_by_indicator)

,indicator,obs_days_count
1,1.192.18.4,18
8,100.27.42.247,18
16,101.89.148.7,43
17,101.89.174.236,127
19,102.0.5.152,8
...,...,...
5071,www.deepseek.com,221
5072,www.deepseek.com.cdn.cloudflare.net,176
5103,www.prosinthecity.com,13
5108,www.sthda.com,247


In [5]:
import re

# helper to flatten list-or-str into a list of strings
def flatten_hosts(hosts):
    if isinstance(hosts, list):
        return [h for h in hosts if isinstance(h, str)]
    elif isinstance(hosts, str):
        return [hosts]
    return []

# regex: 
# ^[A-Za-z]+       → first label is one “word” (letters only)
# (?:\.[^.]+){3}$  → exactly three more “.” followed by non-dot characters
four_label_word_prefix = re.compile(r'^[A-Za-z]+(?:\.[^.]+){3}$')

# Try to match on 'enrich_org' instead of 'enrich_hostNames'
if 'enrich_org' in recent_tags.columns:
    global matched_results
    exploded = recent_tags.explode('enrich_org')

    mask = exploded['enrich_org'].astype(str).str.match(four_label_word_prefix, na=False)

    matched = recent_tags.loc[exploded[mask].index.unique()]
    matched_results = matched[['indicator','enrich_org']]
else:
    matched = pd.DataFrame(columns=recent_tags.columns)
    matched_results = None


display(matched_results)


,indicator,enrich_org


In [6]:
def has_false_positive(tag_str):
    if not isinstance(tag_str, str):
        return False
    tags = [t.strip().lower() for t in tag_str.split(",")]
    return any("false positive" in t for t in tags)

false_positive_indicators = recent_tags[
    recent_tags['tag_name'].apply(has_false_positive)
]

display(false_positive_indicators[['indicator', 'tag_name']])


,indicator,tag_name


In [ ]:
import pandas as pd
import numpy as np

df = recent_tags.copy()

# ── Severity Binning ────────────────────────────
scoring_bins = [0, 200, 500, 800, 1001]
label_bins = ['low', 'medium', 'high', 'critical']

# ── Feature Weights ─────────────────────────────
Weights = {
    "MALICIOUS_WEIGHT": 2.00,
    "OBSERVATION_COUNT_WEIGHT": 0.02,
    "CONTINUITY_WEIGHT": 0.10,
    "BOTNET_FLAG_WEIGHT": -0.20, 
    "TC_RATING": 0.05,
    "TC_CONFIDENCE": 0.02,
}

# ── Continuity Map ──────────────────────────────
CONTINUITY_MAP = {
    'Address': 1, 'IPv4': 1, 'IPv6': 1,
    'Domain': 2, 'Host': 2, 'URL': 2,
    'EmailAddress': 2, 'EmailSubject': 2,
    'SHA1': 3, 'SHA256': 3, 'MD5': 3
}

# Known feature maximums (absolute, domain-informed)
VT_MAX = 94                  # max VT malicious detections
MAX_OBS_REALISTIC = 1000     # max expected sightings in 365 days
MAX_RATING = 5               # ThreatConnect rating scale (0–5)
MAX_CONFIDENCE = 100         # ThreatConnect confidence scale (0–100)
BOTNET_MULTIPLIER = 0.4      # penalize 60%
FALSE_POSITIVE_WEIGHT = 0.9  # penalize 10%
# ── Raw weighted contributions ──────────────────
df['malicious_scaled'] = np.log1p(df['enrich_vtMaliciousCount'])
df['malicious_raw_score'] = df['malicious_scaled'] * Weights['MALICIOUS_WEIGHT']

df['obs_count_raw_score'] = df['obs_count'] * Weights['OBSERVATION_COUNT_WEIGHT']

df['continuity_val'] = df['type'].map(CONTINUITY_MAP).fillna(0)
df['continuity_raw_score'] = df['continuity_val'] * Weights['CONTINUITY_WEIGHT']

df['tc_raw_rating'] = df['rating'] * Weights['TC_RATING']
df['tc_raw_confidence'] = df['confidence'] * Weights['TC_CONFIDENCE']

# ── Botnet penalty ─────────────────────────────
botnet_actions = {'Scanning','DDoS','Spam','Phishing','Cryptojacking','Credential Stuffing'}
col = df.get('Botnet', None)
if col is None:
    df['botnet_flag'] = 0
else:
    def is_botnet(val):
        if isinstance(val, (list, set, tuple)):
            return int(bool(set(map(str, val)) & botnet_actions))
        if isinstance(val, str):
            parts = {p.strip() for p in val.replace(',', ' ').split()}
            return int(bool(parts & botnet_actions))
        return 0
    df['botnet_flag'] = col.apply(is_botnet).astype(int)

# ── Combine into total raw score ───────────────
df['raw_score'] = (
    df['malicious_raw_score'] +
    df['obs_count_raw_score'] +
    df['continuity_raw_score'] +
    df['tc_raw_rating'] +
    df['tc_raw_confidence']
)

# ── False Positive penalty ─────────────────────────────
if 'falsePositives' in df.columns:
    FALSE_POSITIVE_MULTIPLIER = FALSE_POSITIVE_WEIGHT  # rename for clarity
    df['falsePositives'] = pd.to_numeric(df['falsePositives'], errors='coerce').fillna(0)
    
    df['raw_score'] = np.where(
        df['falsePositives'] > 0,
        df['raw_score'] * FALSE_POSITIVE_MULTIPLIER,
        df['raw_score']
    )



df['raw_score'] = np.where(df['botnet_flag'] == 1,
                           df['raw_score'] * BOTNET_MULTIPLIER,
                           df['raw_score'])

# ── Absolute Cap (stand-alone scoring) ─────────
RAW_SCORE_CAP = (
    np.log1p(VT_MAX) * Weights['MALICIOUS_WEIGHT'] +
    MAX_OBS_REALISTIC * Weights['OBSERVATION_COUNT_WEIGHT'] +
    max(CONTINUITY_MAP.values()) * Weights['CONTINUITY_WEIGHT'] +
    (MAX_RATING * Weights['TC_RATING']) +
    (MAX_CONFIDENCE * Weights['TC_CONFIDENCE'])
)

# ── Normalize to 0–1000 scale ──────────────────
df['Threat_Score'] = (1000 * (df['raw_score'] / RAW_SCORE_CAP).clip(0, 1)).round().fillna(0).astype(int)

# ── Assign Severity bin ─────────────────────────
df['Severity'] = pd.cut(df['Threat_Score'],
                        bins=scoring_bins,
                        labels=label_bins,
                        right=False)

# ── Human-readable explanation for each score ─────────────────────────────────
def _name_map():
    return {
        'malicious_raw_score': 'VT malicious (log-scaled)',
        'obs_count_raw_score': 'Recent observations',
        'continuity_raw_score': 'Continuity (indicator type)',
        'tc_raw_rating': 'TC rating',
        'tc_raw_confidence': 'TC confidence',
    }

_NAME_MAP = _name_map()

def build_explanation(row):
    # Pull contributions safely
    parts = {
        'malicious_raw_score': float(row.get('malicious_raw_score', 0) or 0),
        'obs_count_raw_score': float(row.get('obs_count_raw_score', 0) or 0),
        'continuity_raw_score': float(row.get('continuity_raw_score', 0) or 0),
        'tc_raw_rating': float(row.get('tc_raw_rating', 0) or 0),
        'tc_raw_confidence': float(row.get('tc_raw_confidence', 0) or 0),
    }
    total = float(row.get('raw_score', 0) or 0)
    score = float(row.get('Threat_Score', 0) or 0)
    sev = str(row.get('Severity', 'nan'))

    # Identify top positive/negative drivers
    contrib_items = sorted(parts.items(), key=lambda kv: abs(kv[1]), reverse=True)
    top = contrib_items[:3]  # top 3 by absolute magnitude

    # Format driver strings with share of total when possible
    driver_bits = []
    for k, v in top:
        label = _NAME_MAP.get(k, k)
        if total != 0:
            pct = 100.0 * (v / total)
            driver_bits.append(f"{label} ({v:+.2f}, {pct:+.1f}% of total)")
        else:
            driver_bits.append(f"{label} ({v:+.2f})")

    # Penalties & flags
    botnet_flag = int(row.get('botnet_flag', 0) or 0)
    botnet_note = ""
    if botnet_flag == 1:
        botnet_note = f"Botnet-related activity detected: applied {int((1 - BOTNET_MULTIPLIER)*100)}% penalty."
    else:
        botnet_note = "No botnet penalty."

    # False positive note (your current code computes 'false_positive_raw_score' but does not apply it)
    fp_cnt = int(row.get('falsePositives', 0) or 0)
    if fp_cnt > 0:
        fp_note = (f"{fp_cnt} false positive tag(s) present; current config computes a "
                   f"{int((1 - FALSE_POSITIVE_WEIGHT)*100)}% penalty preview "
                   f"but does not apply it to the final score.")
    else:
        fp_note = "No false positive tags."

    # Normalization context
    if RAW_SCORE_CAP > 0:
        cap_pct = 100.0 * (total / RAW_SCORE_CAP)
        cap_text = f"Raw score uses {cap_pct:.1f}% of theoretical cap; normalized to {score:.0f}/1000."
    else:
        cap_text = f"Normalized score {score:.0f}/1000."

    # Severity summary
    sev_text = f"Severity: {sev}."

    # Combine
    reason = ( 
        f"{sev_text} Top drivers: " + "; ".join(driver_bits) + ". "
        f"{botnet_note} {fp_note} {cap_text}"
    )
    return reason

df['Explanation'] = df.apply(build_explanation, axis=1)

#drop duplicates
df.drop_duplicates(subset='indicator', inplace=True)

display(df[['indicator','type','enrich_vtMaliciousCount','obs_count',
            'malicious_raw_score','obs_count_raw_score',
            'continuity_raw_score', 'rating', 'tc_raw_rating',
            'tc_raw_confidence','botnet_flag', 'false_positive_raw_score', 'falsePositives',
            'raw_score','Threat_Score','Severity', 'Explanation']])


,indicator,type,enrich_vtMaliciousCount,obs_count,malicious_raw_score,obs_count_raw_score,continuity_raw_score,rating,tc_raw_rating,tc_raw_confidence,botnet_flag,false_positive_raw_score,falsePositives,raw_score,Threat_Score,Severity,Explanation
0,1.192.18.4,Address,4.0,18,3.218876,0.36,0.1,3.0,0.15,1.32,0,5.148876,0,5.148876,163,low,Severity: low. Top drivers: VT malicious (log-...
1,100.27.42.247,Address,0.0,18,0.000000,0.36,0.1,3.0,0.15,1.56,0,2.170000,0,2.170000,69,low,Severity: low. Top drivers: TC confidence (+1....
2,101.89.148.7,Address,11.0,43,4.969813,0.86,0.1,3.0,0.15,1.56,0,7.639813,0,7.639813,241,medium,Severity: medium. Top drivers: VT malicious (l...
3,101.89.174.236,Address,8.0,127,4.394449,2.54,0.1,3.0,0.15,1.34,0,8.524449,0,8.524449,269,medium,Severity: medium. Top drivers: VT malicious (l...
4,102.0.5.152,Address,1.0,8,1.386294,0.16,0.1,4.0,0.20,1.32,1,3.166294,0,1.266518,40,low,Severity: low. Top drivers: VT malicious (log-...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3890,click.email.active.com,Stripped URL,0.0,103,0.000000,2.06,0.0,1.0,0.05,0.10,0,2.210000,0,2.210000,70,low,Severity: low. Top drivers: Recent observation...
3902,gearaficionado.com,Stripped URL,0.0,1,0.000000,0.02,0.0,5.0,0.25,2.00,0,2.270000,0,2.270000,72,low,Severity: low. Top drivers: TC confidence (+2....
3903,gobookmart.com,Stripped URL,0.0,20,0.000000,0.40,0.0,5.0,0.25,2.00,0,2.650000,0,2.650000,84,low,Severity: low. Top drivers: TC confidence (+2....
3918,modernstoicism.com,Stripped URL,0.0,12,0.000000,0.24,0.0,5.0,0.25,1.88,0,2.370000,0,2.370000,75,low,Severity: low. Top drivers: TC confidence (+1....


In [ ]:
check again: import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import warnings
import re

warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG: all adjustable weights & parameters live here
# ─────────────────────────────────────────────────────────────────────────────
# Temporal/volume
LAMBDA_DECAY = 0.10              # exponential decay for observations
LOG_CLIP_Q = 0.95                # upper clip quantile for log_decayed_obs
TEMP_Q_LOW = 0.25                # quantile for "light" recent sightings
TEMP_Q_HIGH = 0.75               # quantile for "heavy" recent sightings
BURST_Q_HIGH = 0.75              # quantile for "bursty" pattern

# Sources/trust
SRC_Q_LOW = 0.25                 # few corroborating sources
SRC_Q_HIGH = 0.75                # broad corroboration
SRC_WEIGHT_Q_HIGH = 0.75         # high total trust weight
HIGH_TRUST_THRESHOLD = 7         # owner trust cutoff for "high"

# False positives
FP_Q_LOW = 0.25
FP_Q_HIGH = 0.75
FP_RATE_LOW_FLAG = 0.10
FP_RATE_HIGH_FLAG = 0.50

# Context
PORT_Q_HIGH = 0.75
VT_Q_HIGH = 0.75                 # (used only in messaging)
HOST_PENALTY_WEIGHT = 0.15       # fractional downweight applied if pattern matches (e.g., 0.15 = -15%)

# Analyst signals
RATING_HIGH = 4                  # rating considered "high"
CONFIDENCE_HIGH = 80             # confidence considered "high"

# Continuity mapping
CONTINUITY_MAP = {
    'Address': 1, 'IPv4': 1, 'IPv6': 1,
    'Domain': 2, 'Host': 2, 'URL': 2,
    'EmailAddress': 2, 'EmailSubject': 2,
    'SHA1': 3, 'SHA256': 3, 'MD5': 3
}

# Modeling (supervised)
FEATURE_COLS = [
    'continuity_score', 'log_decayed_obs', 'obs_trend_slope', 'obs_burstiness',
    'num_distinct_sources', 'sum_source_weight', 'fp_rate',
    'port_diversity', 'vt_malicious_count', 'port_scan_confidence',
    'botnet_flag', 'Xcrit', 'host_generic_4label_wordprefix'
]
SUPERVISED_RANDOM_STATE = 42
N_FOLDS = 5
TRAINING_TARGET_COL = 'label_score'   # set this to your real 0..1000 label column if available

# Monotonic constraints (optional): +1 increasing, -1 decreasing, 0 unconstrained
USE_MONOTONIC_CONSTRAINTS = True
MONO_CST_BY_FEATURE = {
    'continuity_score': 0,
    'log_decayed_obs': +1,
    'obs_trend_slope': +1,
    'obs_burstiness': +1,            # set 0 if you don't want to enforce
    'num_distinct_sources': +1,
    'sum_source_weight': +1,
    'fp_rate': -1,
    'port_diversity': +1,
    'vt_malicious_count': +1,
    'port_scan_confidence': +1,
    'botnet_flag': +1,
    'Xcrit': +1,
    'host_generic_4label_wordprefix': -1,
}

# Severity bins (do not change unless ranges are updated)
BINS_1000 = [0, 199, 500, 800, 1000]
BIN_LABELS = ['Low', 'Medium', 'High', 'Critical']

# Explanation composition
EXPLAIN_COLS = [
    'msg_temporal','msg_sources','msg_fp','msg_context',
    'msg_behavior','msg_continuity','msg_botnet','msg_xcrit',
    'msg_hostnames','msg_model','msg_final'
]

# Hostname pattern regex (letters-only first label + exactly 3 more labels → total 4)
FOUR_LABEL_WORD_PREFIX = re.compile(r'^[A-Za-z]+(?:\.[^.]+){3}$')


# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────
def flatten_hosts(hosts):
    if isinstance(hosts, list):
        return [h for h in hosts if isinstance(h, str)]
    elif isinstance(hosts, str):
        return [hosts]
    return []


def _mm(s: pd.Series) -> pd.Series:
    """Safe min-max normalization to [0,1]."""
    s = pd.to_numeric(s, errors='coerce').fillna(0).astype(float)
    mn, mx = float(s.min()), float(s.max())
    if mx <= mn:
        return pd.Series(0.0, index=s.index)
    return (s - mn) / (mx - mn)


# ─────────────────────────────────────────────────────────────────────────────
# 1) Temporal & Volume Features
# ─────────────────────────────────────────────────────────────────────────────
def compute_temporal_volume_features(df, daily_counts_df, lambda_decay=None, today=None):
    if today is None:
        today = pd.Timestamp(datetime.utcnow().date())
    if lambda_decay is None:
        lambda_decay = LAMBDA_DECAY

    df = df.copy()
    df['dateAdded'] = pd.to_datetime(df['dateAdded'], errors='coerce', utc=True).dt.tz_convert(None).dt.normalize()
    df['lastObserved'] = pd.to_datetime(df['lastObserved'], errors='coerce', utc=True).dt.tz_convert(None).dt.normalize()

    df['observations'] = pd.to_numeric(df.get('observations', 0), errors='coerce').fillna(0).clip(lower=0)
    df['raw_obs'] = df['observations']

    days = (today - df['dateAdded']).dt.days
    df['days_since_first_seen'] = days.where(days.notna(), 0).clip(lower=0).astype(int)

    df['decayed_obs'] = df['raw_obs'] * np.exp(-lambda_decay * df['days_since_first_seen'])
    df['log_decayed_obs'] = np.log1p(df['decayed_obs'])
    clip_val = df['log_decayed_obs'].quantile(LOG_CLIP_Q)
    df['log_decayed_obs'] = df['log_decayed_obs'].clip(upper=clip_val)

    # daily counts typing
    if daily_counts_df is not None and not daily_counts_df.empty:
        dcd = daily_counts_df.copy()
        dcd['date'] = pd.to_datetime(dcd['date'], errors='coerce')
        dcd['count'] = pd.to_numeric(dcd['count'], errors='coerce').fillna(0)
    else:
        dcd = pd.DataFrame(columns=['indicator','date','count'])

    def trend_burst(ind):
        counts = dcd.loc[dcd['indicator'] == ind].sort_values('date')['count'].values
        if counts.size >= 2:
            x = np.arange(counts.size)
            slope = float(np.polyfit(x, counts, 1)[0])
            mean = counts.mean()
            burst = float(np.std(counts) / mean) if mean > 0 else 0.0
        else:
            slope, burst = 0.0, 0.0
        return slope, burst

    tb = df['indicator'].apply(trend_burst)
    df[['obs_trend_slope', 'obs_burstiness']] = pd.DataFrame(tb.tolist(), index=df.index)

    # messages
    q_obs = df['log_decayed_obs'].quantile([TEMP_Q_LOW, TEMP_Q_HIGH]).to_dict()
    q_burst = df['obs_burstiness'].quantile([BURST_Q_HIGH]).to_dict()
    def msg_temporal(r):
        msgs = []
        if r['log_decayed_obs'] >= q_obs.get(TEMP_Q_HIGH, 0):
            msgs.append("heavy recent sightings (high decayed volume)")
        elif r['log_decayed_obs'] <= q_obs.get(TEMP_Q_LOW, 0):
            msgs.append("light recent sightings (low decayed volume)")
        if r['obs_trend_slope'] > 0:
            msgs.append("upward trend in daily sightings")
        elif r['obs_trend_slope'] < 0:
            msgs.append("downward trend in daily sightings")
        if r['obs_burstiness'] >= q_burst.get(BURST_Q_HIGH, np.inf):
            msgs.append("bursty observation pattern")
        return "; ".join(msgs) or "stable/average observation pattern"
    df['msg_temporal'] = df.apply(msg_temporal, axis=1)
    return df


# ─────────────────────────────────────────────────────────────────────────────
# 2) Source Diversity & Trust
# ─────────────────────────────────────────────────────────────────────────────
def compute_source_diversity(df, source_sightings_df, trust_map):
    df = df.copy()
    ssd = source_sightings_df.copy() if source_sightings_df is not None else pd.DataFrame()
    if ssd.empty:
        df['num_distinct_sources'] = 0
        df['sum_source_weight'] = 0
        df['max_source_weight'] = 0
        df['high_low_trust_ratio'] = 0.0
        df['msg_sources'] = "no corroborating sources"
        return df

    grp = ssd.groupby('indicator', sort=False)
    df['num_distinct_sources'] = grp['ownerId'].nunique().reindex(df['indicator']).fillna(0).astype(int).values

    owner_lists = grp['ownerId'].apply(list).to_dict()
    sums, maxs, ratios = [], [], []
    for ind in df['indicator']:
        owners = owner_lists.get(ind, [])
        weights = [trust_map.get(o, 5) for o in owners]
        sums.append(sum(weights))
        maxs.append(max(weights) if weights else 0)
        high = sum(w >= HIGH_TRUST_THRESHOLD for w in weights)
        low = sum(w < HIGH_TRUST_THRESHOLD for w in weights)
        ratios.append(high / (low if low > 0 else 1))
    df['sum_source_weight'] = sums
    df['max_source_weight'] = maxs
    df['high_low_trust_ratio'] = ratios

    q_src = df['num_distinct_sources'].quantile([SRC_Q_LOW, SRC_Q_HIGH]).to_dict()
    q_weight = df['sum_source_weight'].quantile([SRC_WEIGHT_Q_HIGH]).to_dict()

    def msg_sources(r):
        msgs = []
        if r['num_distinct_sources'] >= q_src.get(SRC_Q_HIGH, np.inf):
            msgs.append("broad corroboration across sources")
        elif r['num_distinct_sources'] <= q_src.get(SRC_Q_LOW, -np.inf):
            msgs.append("limited corroboration")
        if r['sum_source_weight'] >= q_weight.get(SRC_WEIGHT_Q_HIGH, np.inf):
            msgs.append("supported by high‑trust owners")
        if r['high_low_trust_ratio'] >= 1.0 and r['num_distinct_sources'] > 0:
            msgs.append("high‑trust > low‑trust ratio")
        return "; ".join(msgs) or "moderate corroboration"

    df['msg_sources'] = df.apply(msg_sources, axis=1)
    return df


# ─────────────────────────────────────────────────────────────────────────────
# 3) False-Positive Signals
# ─────────────────────────────────────────────────────────────────────────────
def compute_fp_signals(df):
    df = df.copy()
    if 'falsePositiveCount' not in df.columns:
        df['falsePositiveCount'] = 0
    df['falsePositiveCount'] = pd.to_numeric(df['falsePositiveCount'], errors='coerce').fillna(0).clip(lower=0)
    obs = df.get('observations', pd.Series(0, index=df.index)).replace(0, 1)
    df['fp_count'] = df['falsePositiveCount']
    df['fp_rate'] = (df['fp_count'] / obs).clip(lower=0)

    q_fp = df['fp_rate'].quantile([FP_Q_LOW, FP_Q_HIGH]).to_dict()

    def msg_fp(r):
        if r['fp_rate'] >= q_fp.get(FP_Q_HIGH, np.inf) and r['fp_rate'] > 0:
            return "elevated false‑positive rate (down‑weights score)"
        if r['fp_rate'] <= q_fp.get(FP_Q_LOW, -np.inf):
            return "low false‑positive rate"
        return "moderate false‑positive rate"

    df['msg_fp'] = df.apply(msg_fp, axis=1)
    return df


# ─────────────────────────────────────────────────────────────────────────────
# 4) Contextual Metadata (ports, VT, hostnames)
# ─────────────────────────────────────────────────────────────────────────────
def compute_context_metadata(df):
    df = df.copy()
    types = df.get('type')
    if types is not None:
        df = pd.concat([df, pd.get_dummies(types, prefix='type', dtype=int)], axis=1)

    ports = df.get('enrich_openPorts')
    if ports is None:
        df['port_diversity'] = 0
    else:
        def port_count(p):
            if isinstance(p, (list, set, tuple)):
                return len(set([str(x) for x in p]))
            if isinstance(p, str):
                return len(set([x.strip() for x in p.split(',') if x.strip()]))
            return 0
        df['port_diversity'] = ports.apply(port_count).astype(int)

    df['vt_malicious_count'] = pd.to_numeric(df.get('enrich_vtMaliciousCount', 0), errors='coerce').fillna(0).clip(lower=0).astype(int)

    q_port = df['port_diversity'].quantile([PORT_Q_HIGH]).to_dict()
    q_vt = df['vt_malicious_count'].quantile([VT_Q_HIGH]).to_dict()

    def msg_context(r):
        msgs = []
        if r['port_diversity'] >= q_port.get(PORT_Q_HIGH, 0) and r['port_diversity'] > 0:
            msgs.append("broad port activity observed")
        if r['vt_malicious_count'] >= q_vt.get(VT_Q_HIGH, 0) and r['vt_malicious_count'] > 0:
            msgs.append("strong VT malicious consensus")
        elif r['vt_malicious_count'] == 0:
            msgs.append("no VT malicious detections")
        return "; ".join(msgs) or "limited contextual signals"

    df['msg_context'] = df.apply(msg_context, axis=1)

    # Hostname pattern
    def host_match(hosts):
        for h in flatten_hosts(hosts):
            if FOUR_LABEL_WORD_PREFIX.match(h):
                return True
        return False

    host_col = df.get('enrich_hostNames', None)
    if host_col is None:
        df['host_generic_4label_wordprefix'] = 0
        df['msg_hostnames'] = "no hostnames"
    else:
        df['host_generic_4label_wordprefix'] = host_col.apply(host_match).astype(int)
        df['msg_hostnames'] = np.where(
            df['host_generic_4label_wordprefix'] == 1,
            "hostname pattern suggests generic 4‑label prefix (down‑weighted)",
            "no generic 4‑label hostname pattern"
        )
    return df


# ─────────────────────────────────────────────────────────────────────────────
# 5) Behavioral Signals
# ─────────────────────────────────────────────────────────────────────────────
def compute_behavioral(df):
    df = df.copy()
    tags = df.get('enrich_tags')
    if tags is None:
        df['port_scan_confidence'] = 0
    else:
        def has_scanner(x):
            if isinstance(x, (list, set, tuple)):
                s = ' '.join(map(str, x))
            else:
                s = str(x)
            return int('scanner' in s.lower())
        df['port_scan_confidence'] = tags.apply(has_scanner).astype(int)

    df['msg_behavior'] = np.where(df['port_scan_confidence'] > 0,
                                  "scanner behavior present",
                                  "no scanner behavior")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# 6) Continuity Score
# ─────────────────────────────────────────────────────────────────────────────
def compute_continuity_score(df):
    df = df.copy()
    df['continuity_score'] = df.get('type', '').map(CONTINUITY_MAP).fillna(0).astype(int)

    def msg_cont(r):
        if r['continuity_score'] == 3:
            return "hash‑level continuity (long‑lived indicator)"
        if r['continuity_score'] == 2:
            return "name/url continuity (medium‑lived indicator)"
        if r['continuity_score'] == 1:
            return "IP continuity (short‑lived indicator)"
        return "unknown continuity"

    df['msg_continuity'] = df.apply(msg_cont, axis=1)
    return df


# ─────────────────────────────────────────────────────────────────────────────
# 7) Botnet Flag
# ─────────────────────────────────────────────────────────────────────────────
def compute_botnet_flag(df):
    botnet_actions = {'Scanning','DDoS','Spam','Phishing','Cryptojacking','Credential Stuffing'}
    df = df.copy()
    col = df.get('Botnet', None)

    if col is None:
        df['botnet_flag'] = 0
    else:
        def is_botnet(val):
            if isinstance(val, (list, set, tuple)):
                return int(bool(set(map(str, val)) & botnet_actions))
            if isinstance(val, str):
                parts = {p.strip() for p in val.replace(',', ' ').split()}
                return int(bool(parts & botnet_actions))
            return 0
        df['botnet_flag'] = col.apply(is_botnet).astype(int)

    df['msg_botnet'] = np.where(df['botnet_flag'] > 0,
                                "botnet‑style activity reported",
                                "no botnet activity")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# Feature Assembly
# ─────────────────────────────────────────────────────────────────────────────
def compute_features(df_raw, daily_counts_df, source_sightings_df, trust_map):
    df = compute_temporal_volume_features(df_raw, daily_counts_df)
    df = compute_source_diversity(df, source_sightings_df, trust_map)
    df = compute_fp_signals(df)
    df = compute_context_metadata(df)
    df = compute_behavioral(df)
    df = compute_continuity_score(df)
    df = compute_botnet_flag(df)

    rating = pd.to_numeric(df.get('rating', np.nan), errors='coerce').clip(0, 5).fillna(0)
    conf = pd.to_numeric(df.get('confidence', np.nan), errors='coerce').clip(0, 100).fillna(0)
    # Map to [-2, 0] range then average, as in prior logic
    df['Xcrit'] = (((rating/5*2 - 2) + (conf/100*2 - 2)) / 2).astype(float)

    def msg_xcrit(r):
        msgs = []
        try:
            if float(r.get('rating', 0)) >= RATING_HIGH:
                msgs.append("high analyst rating")
            if float(r.get('confidence', 0)) >= CONFIDENCE_HIGH:
                msgs.append("high confidence")
        except Exception:
            pass
        if not msgs:
            msgs.append("neutral analyst signals")
        return "; ".join(msgs)

    df['msg_xcrit'] = df.apply(msg_xcrit, axis=1)
    return df


# ─────────────────────────────────────────────────────────────────────────────
# Pseudo-labels (teacher) when you don't have real labels yet (0..1000)
# ─────────────────────────────────────────────────────────────────────────────
def build_pseudo_labels(df_feats: pd.DataFrame) -> pd.Series:
    v_vt = _mm(df_feats.get('vt_malicious_count', 0))
    v_vol = _mm(df_feats.get('log_decayed_obs', 0))
    v_src_n = _mm(df_feats.get('num_distinct_sources', 0))
    v_src_w = _mm(df_feats.get('sum_source_weight', 0))
    v_ports = _mm(df_feats.get('port_diversity', 0))
    v_fp = _mm(df_feats.get('fp_rate', 0))
    v_scan = pd.to_numeric(df_feats.get('port_scan_confidence', 0), errors='coerce').fillna(0).clip(0,1)
    v_bot = pd.to_numeric(df_feats.get('botnet_flag', 0), errors='coerce').fillna(0).clip(0,1)
    v_host = pd.to_numeric(df_feats.get('host_generic_4label_wordprefix', 0), errors='coerce').fillna(0).clip(0,1)

    # weights: VT medium-strong, volume medium, sources moderate, behavior small
    combined = (
        0.35 * v_vt +
        0.25 * v_vol +
        0.15 * v_src_n +
        0.10 * v_src_w +
        0.05 * v_ports +
        0.05 * v_scan +
        0.05 * v_bot
    )

    # soft penalties
    penalty_fp = (1.0 - 0.5 * v_fp)                       # higher FP lowers score
    penalty_host = (1.0 - HOST_PENALTY_WEIGHT * v_host)   # your 4-label host penalty
    combined = (combined * penalty_fp * penalty_host).clip(0, 1)

    pseudo_1000 = np.rint(combined * 1000).astype(int).clip(0, 1000)
    return pseudo_1000


# ─────────────────────────────────────────────────────────────────────────────
# Train supervised GBDT to predict score_1000 directly (credit-score style)
# ─────────────────────────────────────────────────────────────────────────────
def _monotonic_array():
    arr = []
    for col in FEATURE_COLS:
        arr.append(MONO_CST_BY_FEATURE.get(col, 0))
    return np.array(arr, dtype=int)


def train_supervised_gbr(df_feats: pd.DataFrame) -> HistGradientBoostingRegressor:
    # Build X
    X = (
        df_feats.reindex(columns=FEATURE_COLS)
        .fillna(0)
        .astype(float)
        .values
    )
    n_samples = X.shape[0]

    # Build y
    if TRAINING_TARGET_COL in df_feats.columns:
        y = (
            pd.to_numeric(df_feats[TRAINING_TARGET_COL], errors='coerce')
            .fillna(0)
            .clip(0, 1000)
            .values
        )
        label_source = 'real labels'
    else:
        y = build_pseudo_labels(df_feats).values
        label_source = 'pseudo-labels'

    # Model params (disable early stopping for small data to be safe)
    params = dict(
        random_state=SUPERVISED_RANDOM_STATE,
        learning_rate=0.05,
        max_iter=400,
        max_depth=None,
        l2_regularization=0.0,
        early_stopping=False if n_samples < 50 else 'auto',
    )
    if USE_MONOTONIC_CONSTRAINTS:
        params['monotonic_cst'] = _monotonic_array()

    model = HistGradientBoostingRegressor(**params)

    # ── Optional CV sanity check ──────────────────────────────────────────────
    # Only run if we have at least 2 samples and at least 2 folds possible
    try:
        # Prefer explicit N_FOLDS if defined; otherwise default to 5
        n_folds_requested = N_FOLDS if 'N_FOLDS' in globals() else 5
        n_splits = min(n_folds_requested, n_samples)

        if n_samples >= 2 and n_splits >= 2:
            kf = KFold(n_splits=n_splits, shuffle=True, random_state=SUPERVISED_RANDOM_STATE)
            maes = []
            for tr, te in kf.split(X):
                model.fit(X[tr], y[tr])
                pred = model.predict(X[te]).clip(0, 1000)
                maes.append(mean_absolute_error(y[te], pred))
            print(f"Supervised GBR ({label_source}) CV MAE [{n_splits} folds]: {np.mean(maes):.2f}")
        else:
            print(f"Supervised GBR ({label_source}) — skipped CV (only {n_samples} sample(s)).")
    except Exception as e:
        # Never let CV kill training during quick tests
        print(f"[WARN] CV skipped due to error: {e}")

    # ── Final fit on all data ────────────────────────────────────────────────
    model.fit(X, y)
    return model



# ─────────────────────────────────────────────────────────────────────────────
# Predict with supervised model and map to bins
# ─────────────────────────────────────────────────────────────────────────────
def predict_supervised_gbr(model: HistGradientBoostingRegressor, df_feats: pd.DataFrame) -> pd.DataFrame:
    X = df_feats.reindex(columns=FEATURE_COLS).fillna(0).astype(float).values
    raw = model.predict(X).clip(0, 1000)

    preds = calibrate_scores_piecewise(
        raw,
        q_points=(0.10, 0.50, 0.90, 0.99),
        y_targets=(250, 550, 820, 980),
        df_feats=df_feats
    )

    preds *= (1 - 0.5 * _mm(df_feats['fp_rate']))
    preds *= (1 - HOST_PENALTY_WEIGHT * df_feats['host_generic_4label_wordprefix'])

    analyst_high = (
        (df_feats.get('rating', 0) >= RATING_HIGH) &
        (df_feats.get('confidence', 0) >= CONFIDENCE_HIGH)
    )
    preds = np.where(analyst_high, np.maximum(preds, 600), preds)

    out = df_feats.copy()
    out['score_1000'] = np.rint(preds).astype(int).clip(0, 1000)
    out['severity'] = pd.cut(out['score_1000'], bins=BINS_1000, labels=BIN_LABELS,
                             include_lowest=True, right=True)

    # Quartiles for score and features
    q_score = out['score_1000'].quantile([0.25, 0.75]).to_dict()
    q_feats = {
        'vt_75': out['vt_malicious_count'].quantile(0.75),
        'vol_75': out['log_decayed_obs'].quantile(0.75),
        'src_75': out['num_distinct_sources'].quantile(0.75)
    }

    def msg_model_sup(r):
        if r['score_1000'] >= q_score.get(0.75, 0):
            return "supervised model predicts high score (upper quartile)"
        if r['score_1000'] <= q_score.get(0.25, 0):
            return "supervised model predicts low score (lower quartile)"
        return "supervised model predicts moderate score"
    out['msg_model'] = out.apply(msg_model_sup, axis=1)

    def msg_final(r):
        parts = [f"score={int(r.get('score_1000', 0))}", f"bin={r.get('severity')}"]
        reasons = []
        vt   = float(r.get('vt_malicious_count', 0))
        vol  = float(r.get('log_decayed_obs', 0))
        src  = float(r.get('num_distinct_sources', 0))
        fp   = float(r.get('fp_rate', 0))
        bot  = int(r.get('botnet_flag', 0))
        scan = int(r.get('port_scan_confidence', 0))
        host_pen = int(r.get('host_generic_4label_wordprefix', 0))

        if vt >= q_feats['vt_75'] and vt > 0: reasons.append("strong VT signal")
        elif vt > 0: reasons.append("some VT signal")
        if vol >= q_feats['vol_75']: reasons.append("heavy recent sightings")
        if src >= q_feats['src_75']: reasons.append("broad corroboration")
        if scan: reasons.append("scanner behavior")
        if bot: reasons.append("botnet activity")
        if fp >= FP_RATE_HIGH_FLAG: reasons.append("high FP rate")
        elif fp <= FP_RATE_LOW_FLAG: reasons.append("low FP rate")
        if host_pen: reasons.append(f"hostname pattern penalty −{int(HOST_PENALTY_WEIGHT*100)}%")
        if not reasons: reasons.append("mixed factors")

        sev = str(r.get('severity'))
        return " | ".join(parts) + " || " + f"{sev} severity — driven by {', '.join(reasons)}."
    out['msg_final'] = out.apply(msg_final, axis=1)
    return out

def calibrate_scores_piecewise(preds: np.ndarray,
                               q_points=(0.10, 0.50, 0.90, 0.99),
                               y_targets=(250, 550, 820, 980),
                               lo=0, hi=1000,
                               df_feats: pd.DataFrame=None) -> np.ndarray:
    preds = np.asarray(preds, dtype=float)
    xs = np.quantile(preds, q_points)
    ys = np.array(y_targets, dtype=float)

    # Fallback if preds are nearly constant
    if np.allclose(xs, xs[0]):
        scale = (hi - lo) / max(1.0, preds.max() - preds.min())
        out = lo + (preds - preds.min()) * scale
    else:
        out = np.interp(preds, xs, ys)

    out = np.clip(out, lo, hi)

    # Apply malicious-aware floor
    if df_feats is not None:
        strong = (
            (df_feats['vt_malicious_count'] > 0) |
            (df_feats['num_distinct_sources'] > 2) |
            (df_feats['port_scan_confidence'] > 0) |
            (df_feats['botnet_flag'] > 0)
        )
        out = np.where(strong, np.maximum(out, 400), out)  # at least "Medium"

    return out





# ─────────────────────────────────────────────────────────────────────────────
# Assemble explanations
# ─────────────────────────────────────────────────────────────────────────────
def assemble_explanations(df):
    df = df.copy()
    def join_msgs(row):
        msgs = [str(row[c]) for c in EXPLAIN_COLS if c in row and pd.notna(row[c]) and str(row[c]).strip()]
        return " ; ".join(msgs)
    df['explain'] = df.apply(join_msgs, axis=1)
    return df


# ─────────────────────────────────────────────────────────────────────────────
# Example Main (uses CONFIG values)
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    # Expecting a DataFrame `recent_tags` in your environment.
    try:
        df_raw = recent_tags  # assumed provided by caller's environment
    except NameError:
        # Fallback: create a tiny synthetic dataset for demo purposes
        df_raw = pd.DataFrame({
            'indicator': [f'1.2.3.{i}' for i in range(1, 11)],
            'dateAdded': pd.Timestamp.utcnow().normalize(),
            'lastObserved': pd.Timestamp.utcnow().normalize(),
            'observations': np.random.poisson(5, 10),
            'falsePositiveCount': np.random.binomial(5, 0.2, 10),
            'type': ['IPv4']*10,
            'enrich_vtMaliciousCount': np.random.randint(0, 25, 10),
            'enrich_openPorts': [list(np.random.choice([80,443,22,445], size=np.random.randint(0,4), replace=False)) for _ in range(10)],
            'enrich_tags': [['scanner'] if np.random.rand()>0.6 else [] for _ in range(10)],
            'enrich_hostNames': [['mail.example.com.a'] for _ in range(10)],
            'rating': np.random.randint(0,6,10),
            'confidence': np.random.randint(0,101,10),
            'Botnet': ['Scanning' if np.random.rand()>0.7 else '' for _ in range(10)],
            'ownerId': np.random.choice(['A','B','C'], size=10)
        })

    today = pd.Timestamp(datetime.utcnow().date())
    dates = [today - pd.Timedelta(days=i) for i in range(7)]

    # Build synthetic daily counts (replace with real in production)
    daily = [{'indicator': ind, 'date': d, 'count': np.random.poisson(5)}
             for ind in df_raw['indicator'] for d in dates]
    daily_counts_df = pd.DataFrame(daily)

    # Example source sightings (replace with real per-indicator owners)
    owner_ids = df_raw.get('ownerId', pd.Series([], dtype=object))
    unique_owners = pd.Series(owner_ids).dropna().unique()
    if len(unique_owners) > 0:
        source_sightings_df = pd.DataFrame(
            [{'indicator': ind, 'ownerId': oid}
             for ind in df_raw['indicator'] for oid in unique_owners]
        )
        trust_map = {oid: int(np.random.choice([1,5,7,10])) for oid in unique_owners}
    else:
        source_sightings_df = pd.DataFrame(columns=['indicator','ownerId'])
        trust_map = {}

    # 1) Features
    df_feats = compute_features(df_raw, daily_counts_df, source_sightings_df, trust_map)

    # 2) Train supervised GBR (uses real labels if available; else pseudo-labels)
    #sup_model = train_supervised_gbr(df_feats)
    sup_model = train_supervised_gbr(df_feats)

    # 3) Predict and assemble explanations
    df_scored = predict_supervised_gbr(sup_model, df_feats)
    df_result = assemble_explanations(df_scored)

    # Display results
    cols = ['indicator', 'score_1000', 'vt_malicious_count', 'severity', 'explain']
    try:
        from IPython.display import display
        display(df_result[cols])
    except Exception:
        print(df_result[cols].to_string(index=False))
